# E3 · KFold on data sorted by class and `error_score=0` hiding everything

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

Understand why a cross-validation returns an AUC of 0 on all the folds without raising errors, and fix the splitter.
The colleague's code has two defects that hide each other: data sorted by target and `error_score=0`.

## Dataset

`ds.loans(with_missing=False)`: the four numeric columns and the target `defaulted`, but with the rows **sorted by target** (first all the 0s,
then all the 1s), as happens when two extracts are concatenated. Model: logistic regression with scaler.

## Educational purpose

Recognize two behaviors that combine badly: `KFold` without shuffling creates **contiguous** folds, and `error_score=0` turns
errors into scores. The result is a number (0.0) that looks like information but is a hidden error.

## Task

1. Run the colleague's cell: `auc_bug`. Is it 0? Why are there no visible errors?
2. Repeat with `error_score="raise"` inside a `try/except` and save in `message` the first line of the error message (string).
3. Save `bug_positives_per_fold`: the list of the positives (`y_sorted == 1`) in the **validation** fold of each of the 5 folds of `KFold(5)`.
4. Fix it with `StratifiedKFold(5, shuffle=True, random_state=0)` **without** `error_score`: save `auc_ok` and `ok_positives_per_fold`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False).sort_values("defaulted").reset_index(drop=True)      # rows sorted by target
X_sorted = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y_sorted = df["defaulted"].to_numpy()
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

## Your work

### The colleague's code

In [ ]:
bug_scores = cross_val_score(model, X_sorted, y_sorted, cv=KFold(5), scoring="roc_auc", error_score=0)
auc_bug = bug_scores.mean()
print("AUC per fold:", bug_scores.round(3), "| mean:", round(auc_bug, 3))

### Your diagnosis

In [ ]:
message = None
bug_positives_per_fold = None
auc_ok = auc_bug
ok_positives_per_fold = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`try: cross_val_score(model, X_sorted, y_sorted, cv=KFold(5), scoring="roc_auc", error_score="raise") except ValueError as e: message = str(e).splitlines()[0]`. Positives: `[int(y_sorted[va].sum()) for _, va in KFold(5).split(X_sorted)]`.'):
    check(auc_bug == 0.0, "the colleague's number is 0.0 on all the folds: they are errors turned into scores", "don't modify the colleague's code")
    check(isinstance(message, str) and ("class" in message.lower() or "one" in message.lower()),
          "the error message talks about a single class", "with error_score='raise' cross_val_score raises a ValueError about the classes")
    check(bug_positives_per_fold is not None and sorted(bug_positives_per_fold)[:4] == [0, 0, 0, 0] and max(bug_positives_per_fold) > 200,
          "four validation folds have no positives at all, the last one almost only positives", "with KFold without shuffle the folds are contiguous blocks of sorted rows")
    check(0.7 < auc_ok < 0.85, "with stratification the AUC is the true one (between 0.7 and 0.85)", "StratifiedKFold(5, shuffle=True, random_state=0) without error_score")
    check(ok_positives_per_fold is not None and max(ok_positives_per_fold) - min(ok_positives_per_fold) <= 1,
          "with stratification the positives are evenly distributed (difference at most 1)", "every fold must have about 19% positives")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_kfold_on_sorted_data_and_error_score.ipynb`. Open it after you pass the check, or after you have really tried.